# Multi-Site Diurnal Wavelength Analysis

**Purpose**: Extend the Addis Ababa (JACROS) diurnal wavelength analysis to Beijing, Delhi, and JPL to:
1. Diagnose whether the identical IR/blue/red wavelengths at Addis are an instrument issue or site-specific
2. Compare spectral variability (AAE, Delta-C, wavelength ratios) across sites by season
3. Identify if wavelength divergence patterns (like those seen at Queens College) appear at other sites

**Key research question**: Do the other sites show the same collapsed IR/Blue/Red wavelengths as Addis, or is this unique to the ETAD aethalometer?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import gc
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', category=FutureWarning)

# Add project root to path
sys.path.insert(0, str(Path.cwd().parents[1]))
from src.config.multi_site_seasons import (
    SITE_SEASONS, SITE_COLORS, assign_season, get_season_list, get_season_colors
)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 120,
})

# --- Wavelength configuration ---
WAVELENGTHS = {
    'UV':    {'nm': 370, 'color': '#8B00FF'},
    'Blue':  {'nm': 470, 'color': '#0000FF'},
    'Green': {'nm': 520, 'color': '#00AA00'},
    'Red':   {'nm': 660, 'color': '#FF0000'},
    'IR':    {'nm': 880, 'color': '#222222'},
}
BC_COLS = {name: f'{name} BCc' for name in WAVELENGTHS}

# --- Data paths ---
DATA_DIR = Path('/home/user/aethmodular/data')
SITE_DATA_FILES = {
    'Beijing':     DATA_DIR / 'df_cleaned_Beijing_manual_BCc.pkl',
    'Delhi':       DATA_DIR / 'df_cleaned_Delhi_manual_BCc.pkl',
    'JPL':         DATA_DIR / 'df_cleaned_JPL_manual_BCc.pkl',
    'Addis_Ababa': DATA_DIR / 'df_jacros_cleaned_API_and_OG_manual_BC_all_wl.pkl',
}

# Output directory
PLOT_DIR = Path('output/plots/multisite_wavelength')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete. Sites:", list(SITE_DATA_FILES.keys()))

## 1. Data Loading & Helper Functions

In [ ]:
# ===================================================================
# Data loading — subset to needed columns only for memory efficiency
# ===================================================================

def load_and_prepare_site(site_name, filepath):
    """Load aethalometer pkl, subset columns, QC, convert units, assign seasons."""
    print(f"  Loading {site_name} from {filepath.name}...")
    
    # Read full pkl then immediately subset
    df_full = pd.read_pickle(filepath)
    keep = ['datetime_local'] + [c for c in BC_COLS.values() if c in df_full.columns]
    df = df_full[keep].copy()
    del df_full
    gc.collect()
    
    # Set datetime index
    df['datetime_local'] = pd.to_datetime(df['datetime_local'])
    df = df.set_index('datetime_local').sort_index()
    
    # Convert ng/m³ → µg/m³
    bc_existing = [c for c in BC_COLS.values() if c in df.columns]
    df[bc_existing] = df[bc_existing] / 1000.0
    
    # QC: remove negatives
    for c in bc_existing:
        df.loc[df[c] < 0, c] = np.nan
    
    # Time features
    df['Hour'] = df.index.hour
    df['Month'] = df.index.month
    df['Date'] = df.index.date
    
    # Assign seasons
    assign_season(df, site_name)
    
    print(f"    → {len(df):,} rows | {df.index.min().date()} to {df.index.max().date()} "
          f"| Seasons: {df['Season'].nunique()}")
    return df


# ===================================================================
# Derived metrics — AAE, Delta-C, ratios, CV
# ===================================================================

def compute_derived_metrics(df):
    """Add all derived wavelength metrics to the DataFrame."""
    ir_col = BC_COLS['IR']
    uv_col = BC_COLS['UV']
    blue_col = BC_COLS['Blue']
    
    wl_uv = WAVELENGTHS['UV']['nm']
    wl_blue = WAVELENGTHS['Blue']['nm']
    wl_ir = WAVELENGTHS['IR']['nm']
    
    # --- Wavelength / IR ratios ---
    for name in ['UV', 'Blue', 'Green', 'Red']:
        col = BC_COLS[name]
        mask = (df[col].notna()) & (df[ir_col].notna()) & (df[ir_col] > 0.1)
        df[f'{name}/IR'] = np.where(mask, df[col] / df[ir_col], np.nan)
    
    # --- AAE (Absorption Angstrom Exponent) ---
    mask_uv = (df[uv_col] > 0.1) & (df[ir_col] > 0.1)
    df['AAE_UV_IR'] = np.where(
        mask_uv,
        -np.log(df[uv_col] / df[ir_col]) / np.log(wl_uv / wl_ir),
        np.nan
    )
    
    mask_blue = (df[blue_col] > 0.1) & (df[ir_col] > 0.1)
    df['AAE_Blue_IR'] = np.where(
        mask_blue,
        -np.log(df[blue_col] / df[ir_col]) / np.log(wl_blue / wl_ir),
        np.nan
    )
    
    # --- Delta-C (biomass burning tracer) ---
    df['Delta_C'] = df[uv_col] - df[ir_col]
    
    # --- CV across wavelengths ---
    bc_matrix = df[[c for c in BC_COLS.values() if c in df.columns]]
    row_mean = bc_matrix.mean(axis=1)
    row_std = bc_matrix.std(axis=1)
    df['wl_CV'] = (row_std / row_mean).where(row_mean > 0.1)
    
    return df


# ===================================================================
# Plotting helpers
# ===================================================================

def plot_diurnal_wavelength(df, site_name, ax_abs, ax_norm):
    """Plot diurnal BC by wavelength: absolute (left) and normalized to IR (right)."""
    ir_col = BC_COLS['IR']
    
    for name, info in WAVELENGTHS.items():
        col = BC_COLS[name]
        hourly = df.groupby('Hour')[col]
        med = hourly.median()
        q25 = hourly.quantile(0.25)
        q75 = hourly.quantile(0.75)
        
        label = f"{name} ({info['nm']} nm)"
        ax_abs.plot(med.index, med.values, marker='o', ms=3, lw=1.8,
                    color=info['color'], label=label)
        ax_abs.fill_between(med.index, q25, q75, alpha=0.1, color=info['color'])
    
    ax_abs.set_title(f'{site_name} — Absolute BC')
    ax_abs.set_xlabel('Hour of Day')
    ax_abs.set_ylabel('BC (µg/m³)')
    ax_abs.legend(fontsize=8)
    ax_abs.set_xlim(-0.5, 23.5)
    
    # Normalized: divide each wavelength median by IR median at each hour
    ir_med = df.groupby('Hour')[ir_col].median()
    for name, info in WAVELENGTHS.items():
        col = BC_COLS[name]
        med = df.groupby('Hour')[col].median()
        ratio = med / ir_med
        ax_norm.plot(ratio.index, ratio.values, marker='o', ms=3, lw=1.8,
                     color=info['color'], label=f"{name}/{WAVELENGTHS['IR']['nm']}nm")
    
    ax_norm.axhline(1.0, color='gray', ls='--', lw=1, alpha=0.7)
    ax_norm.set_title(f'{site_name} — Normalized to IR')
    ax_norm.set_xlabel('Hour of Day')
    ax_norm.set_ylabel('Ratio (wl / IR)')
    ax_norm.legend(fontsize=8)
    ax_norm.set_xlim(-0.5, 23.5)


def plot_ratio_diurnal(df, site_name, ax):
    """Plot wavelength/IR ratio diurnal patterns."""
    for name in ['UV', 'Blue', 'Green', 'Red']:
        rcol = f'{name}/IR'
        if rcol not in df.columns:
            continue
        hourly = df.groupby('Hour')[rcol]
        med = hourly.median()
        q25 = hourly.quantile(0.25)
        q75 = hourly.quantile(0.75)
        
        ax.plot(med.index, med.values, marker='o', ms=3, lw=1.8,
                color=WAVELENGTHS[name]['color'], label=f'{name}/IR')
        ax.fill_between(med.index, q25, q75, alpha=0.1, color=WAVELENGTHS[name]['color'])
    
    ax.axhline(1.0, color='gray', ls='--', lw=1, alpha=0.7)
    ax.set_title(f'{site_name} — Wavelength/IR Ratios')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Ratio')
    ax.legend(fontsize=8)
    ax.set_xlim(-0.5, 23.5)


def plot_aae_diurnal(df, site_name, ax):
    """Plot AAE diurnal pattern for UV/IR and Blue/IR."""
    for col, label, color in [('AAE_UV_IR', 'AAE (UV/IR)', '#8B00FF'),
                               ('AAE_Blue_IR', 'AAE (Blue/IR)', '#0000FF')]:
        hourly = df.groupby('Hour')[col]
        med = hourly.median()
        q25 = hourly.quantile(0.25)
        q75 = hourly.quantile(0.75)
        
        ax.plot(med.index, med.values, marker='o', ms=3, lw=1.8, color=color, label=label)
        ax.fill_between(med.index, q25, q75, alpha=0.1, color=color)
    
    ax.axhline(1.0, color='gray', ls=':', lw=1, label='Fossil fuel ref')
    ax.axhline(1.5, color='brown', ls=':', lw=1, label='Biomass ref')
    ax.set_title(f'{site_name} — Angstrom Absorption Exponent')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('AAE')
    ax.legend(fontsize=7)
    ax.set_xlim(-0.5, 23.5)


def plot_delta_c_diurnal(df, site_name, ax):
    """Plot Delta-C (UV BCc - IR BCc) diurnal pattern."""
    hourly = df.groupby('Hour')['Delta_C']
    med = hourly.median()
    q25 = hourly.quantile(0.25)
    q75 = hourly.quantile(0.75)
    
    ax.plot(med.index, med.values, 'o-', lw=2.5, ms=5, color='#8B4513')
    ax.fill_between(med.index, q25, q75, alpha=0.2, color='#8B4513')
    ax.axhline(0, color='gray', ls='--', lw=1)
    ax.set_title(f'{site_name} — Delta-C (UV − IR)')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Delta-C (µg/m³)')
    ax.set_xlim(-0.5, 23.5)


def plot_cv_diurnal(df, site_name, ax):
    """Plot wavelength CV diurnal pattern."""
    hourly = df.groupby('Hour')['wl_CV']
    med = hourly.median()
    q25 = hourly.quantile(0.25)
    q75 = hourly.quantile(0.75)
    
    ax.plot(med.index, med.values, 'o-', lw=2.5, ms=5, color='#2C3E50')
    ax.fill_between(med.index, q25, q75, alpha=0.2, color='#2C3E50')
    ax.set_title(f'{site_name} — Wavelength CV')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Coefficient of Variation')
    ax.set_xlim(-0.5, 23.5)


print("Helper functions defined.")

In [ ]:
# Load all sites (sequentially for memory management)
site_data = {}
for site_name, filepath in SITE_DATA_FILES.items():
    df = load_and_prepare_site(site_name, filepath)
    df = compute_derived_metrics(df)
    site_data[site_name] = df
    gc.collect()

print(f"\nAll {len(site_data)} sites loaded successfully.")
for name, df in site_data.items():
    ir_valid = df['IR BCc'].notna().sum()
    print(f"  {name:15s}: {len(df):>10,} rows, {ir_valid:>10,} valid IR BCc, "
          f"median IR = {df['IR BCc'].median():.2f} µg/m³")

---
## 2. Diurnal BC by Wavelength (All Sites)

Median hourly BC concentration for each wavelength, with IQR shading. Left column shows absolute concentrations, right column shows each wavelength normalized to IR. **Key diagnostic**: if IR/Blue/Red overlap at one site but not others, it suggests an instrument issue.

In [ ]:
sites = list(site_data.keys())
fig, axes = plt.subplots(len(sites), 2, figsize=(16, 5 * len(sites)))

for i, site in enumerate(sites):
    plot_diurnal_wavelength(site_data[site], site, axes[i, 0], axes[i, 1])

fig.suptitle('Diurnal BC by Wavelength — All Sites', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'diurnal_wavelength_all_sites.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Wavelength/IR Ratio Diurnal Patterns

Ratios of each wavelength to IR at each hour. A flat line at 1.0 for Blue and Red means those wavelengths are identical to IR (the Addis issue). Divergence from 1.0 indicates spectral variation — expected for different source types.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes_flat = axes.flatten()

for i, site in enumerate(sites):
    plot_ratio_diurnal(site_data[site], site, axes_flat[i])

fig.suptitle('Wavelength/IR Ratio Diurnal — All Sites', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'ratio_diurnal_all_sites.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Angstrom Absorption Exponent (AAE) Diurnal

AAE = −ln(BC₁/BC₂) / ln(λ₁/λ₂). Calculated for UV/IR (primary) and Blue/IR (secondary).
- ~1.0 → fossil fuel dominated
- ~1.5+ → biomass burning influence

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes_flat = axes.flatten()

for i, site in enumerate(sites):
    plot_aae_diurnal(site_data[site], site, axes_flat[i])

fig.suptitle('AAE Diurnal Pattern — All Sites', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'aae_diurnal_all_sites.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Delta-C Diurnal (UV BCc − IR BCc)

Delta-C is a biomass burning tracer. Positive values indicate UV absorption exceeding IR, suggestive of biomass/brown carbon influence.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes_flat = axes.flatten()

for i, site in enumerate(sites):
    plot_delta_c_diurnal(site_data[site], site, axes_flat[i])

fig.suptitle('Delta-C Diurnal — All Sites', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'delta_c_diurnal_all_sites.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Wavelength Coefficient of Variation (CV) Diurnal

CV = std(all wavelengths) / mean(all wavelengths) at each minute. Higher CV indicates greater spectral divergence — wavelengths are reading differently.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes_flat = axes.flatten()

for i, site in enumerate(sites):
    plot_cv_diurnal(site_data[site], site, axes_flat[i])

fig.suptitle('Wavelength CV Diurnal — All Sites', fontsize=15, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'cv_diurnal_all_sites.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Seasonal Breakdown — Diurnal Wavelength by Season (Per Site)

Each site gets its own figure with seasons as columns. Top row: absolute BC by wavelength. Bottom row: wavelength/IR ratios. This is critical — aggregated data hides seasonal patterns.

In [ ]:
# --- Seasonal diurnal wavelength overlay (per site) ---
for site in sites:
    df = site_data[site]
    season_list = get_season_list(site)
    n_seasons = len(season_list)
    
    fig, axes = plt.subplots(2, n_seasons, figsize=(5.5 * n_seasons, 10))
    if n_seasons == 1:
        axes = axes.reshape(2, 1)
    
    for j, season in enumerate(season_list):
        sdata = df[df['Season'] == season]
        n_pts = sdata['IR BCc'].notna().sum()
        
        # Top row: absolute concentrations
        ax = axes[0, j]
        for name, info in WAVELENGTHS.items():
            col = BC_COLS[name]
            med = sdata.groupby('Hour')[col].median()
            ax.plot(med.index, med.values, 'o-', ms=3, lw=1.8,
                    color=info['color'], label=f"{name} ({info['nm']})")
        ax.set_title(f'{season}\n(n={n_pts:,})', fontsize=10)
        ax.set_xlabel('Hour')
        ax.set_ylabel('BC (µg/m³)' if j == 0 else '')
        ax.set_xlim(-0.5, 23.5)
        if j == 0:
            ax.legend(fontsize=7)
        
        # Bottom row: wavelength/IR ratios
        ax2 = axes[1, j]
        for name in ['UV', 'Blue', 'Green', 'Red']:
            rcol = f'{name}/IR'
            if rcol not in sdata.columns:
                continue
            med = sdata.groupby('Hour')[rcol].median()
            ax2.plot(med.index, med.values, 'o-', ms=3, lw=1.8,
                     color=WAVELENGTHS[name]['color'], label=f'{name}/IR')
        ax2.axhline(1.0, color='gray', ls='--', lw=1, alpha=0.7)
        ax2.set_xlabel('Hour')
        ax2.set_ylabel('Ratio' if j == 0 else '')
        ax2.set_xlim(-0.5, 23.5)
        if j == 0:
            ax2.legend(fontsize=7)
    
    fig.suptitle(f'{site} — Seasonal Diurnal Wavelength Analysis', fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'seasonal_diurnal_{site}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# --- Seasonal AAE diurnal (per site) ---
for site in sites:
    df = site_data[site]
    season_list = get_season_list(site)
    season_colors = get_season_colors(site)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    for aae_col, ax, title in [('AAE_UV_IR', axes[0], 'AAE (UV/IR)'),
                                ('AAE_Blue_IR', axes[1], 'AAE (Blue/IR)')]:
        for season in season_list:
            sdata = df[df['Season'] == season]
            med = sdata.groupby('Hour')[aae_col].median()
            ax.plot(med.index, med.values, 'o-', ms=3, lw=1.8,
                    color=season_colors[season], label=season)
        
        ax.axhline(1.0, color='gray', ls=':', lw=1)
        ax.axhline(1.5, color='brown', ls=':', lw=1)
        ax.set_title(f'{site} — {title}')
        ax.set_xlabel('Hour of Day')
        ax.set_ylabel('AAE')
        ax.legend(fontsize=8)
        ax.set_xlim(-0.5, 23.5)
    
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'seasonal_aae_{site}.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 8. Cross-Site Comparison — Key Diagnostic

All four sites overlaid on the same axes. This directly answers: **is the wavelength collapse unique to Addis, or do other sites show it too?**

In [ ]:
# --- Cross-site: UV/IR ratio overlay (key instrument diagnostic) ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: UV/IR ratio for all sites
ax = axes[0]
for site in sites:
    med = site_data[site].groupby('Hour')['UV/IR'].median()
    ax.plot(med.index, med.values, 'o-', ms=4, lw=2,
            color=SITE_COLORS[site], label=site)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_title('UV/IR Ratio — All Sites')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('UV/IR Ratio')
ax.legend(fontsize=9)
ax.set_xlim(-0.5, 23.5)

# Panel 2: Blue/IR ratio for all sites
ax = axes[1]
for site in sites:
    med = site_data[site].groupby('Hour')['Blue/IR'].median()
    ax.plot(med.index, med.values, 'o-', ms=4, lw=2,
            color=SITE_COLORS[site], label=site)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_title('Blue/IR Ratio — All Sites')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Blue/IR Ratio')
ax.legend(fontsize=9)
ax.set_xlim(-0.5, 23.5)

# Panel 3: AAE (UV/IR) for all sites
ax = axes[2]
for site in sites:
    med = site_data[site].groupby('Hour')['AAE_UV_IR'].median()
    ax.plot(med.index, med.values, 'o-', ms=4, lw=2,
            color=SITE_COLORS[site], label=site)
ax.axhline(1.0, color='gray', ls=':', lw=1)
ax.axhline(1.5, color='brown', ls=':', lw=1)
ax.set_title('AAE (UV/IR) — All Sites')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('AAE')
ax.legend(fontsize=9)
ax.set_xlim(-0.5, 23.5)

# Panel 4: Delta-C (normalized by site median IR BCc) for all sites
ax = axes[3]
for site in sites:
    df = site_data[site]
    med_dc = df.groupby('Hour')['Delta_C'].median()
    ir_median = df['IR BCc'].median()
    normalized = med_dc / ir_median if ir_median > 0 else med_dc
    ax.plot(normalized.index, normalized.values, 'o-', ms=4, lw=2,
            color=SITE_COLORS[site], label=site)
ax.axhline(0, color='gray', ls='--', lw=1)
ax.set_title('Normalized Delta-C — All Sites')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Delta-C / median IR BCc')
ax.legend(fontsize=9)
ax.set_xlim(-0.5, 23.5)

fig.suptitle('Cross-Site Wavelength Comparison', fontsize=15, y=1.03)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'cross_site_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Cross-site: Green/IR ratio (the anomalous offset seen at Addis) ---
fig, ax = plt.subplots(figsize=(10, 5))

for site in sites:
    med = site_data[site].groupby('Hour')['Green/IR'].median()
    ax.plot(med.index, med.values, 'o-', ms=4, lw=2,
            color=SITE_COLORS[site], label=site)

ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.set_title('Green/IR Ratio — All Sites\n(Addis shows offset green; do other sites?)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Green/IR Ratio')
ax.legend(fontsize=10)
ax.set_xlim(-0.5, 23.5)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'green_ir_ratio_cross_site.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Representative Day Fingerprints

For each site, pick 3 days: high, medium, and low wavelength CV. Shows how individual days look versus the aggregate.

In [ ]:
def find_representative_days(df, min_minutes=1200):
    """Pick high, medium, low CV days with sufficient data."""
    ir_col = BC_COLS['IR']
    daily_cv = df.groupby('Date')['wl_CV'].median().dropna()
    daily_count = df.groupby('Date')[ir_col].count()
    good_days = daily_count[daily_count >= min_minutes].index
    daily_cv = daily_cv.loc[daily_cv.index.isin(good_days)]
    
    if len(daily_cv) < 10:
        return None
    
    high = daily_cv.nlargest(20).index[min(5, len(daily_cv.nlargest(20)) - 1)]
    low = daily_cv.nsmallest(20).index[min(5, len(daily_cv.nsmallest(20)) - 1)]
    mid_idx = len(daily_cv) // 2
    medium = daily_cv.sort_values().index[mid_idx]
    return {'High CV': high, 'Medium CV': medium, 'Low CV': low}


fig, axes = plt.subplots(len(sites), 3, figsize=(18, 5 * len(sites)))

for i, site in enumerate(sites):
    df = site_data[site]
    rep_days = find_representative_days(df)
    
    if rep_days is None:
        for j in range(3):
            axes[i, j].text(0.5, 0.5, f'{site}\nInsufficient data',
                           ha='center', va='center', transform=axes[i, j].transAxes)
        continue
    
    for j, (label, day) in enumerate(rep_days.items()):
        ax = axes[i, j]
        day_data = df[df['Date'] == day]
        
        for name, info in WAVELENGTHS.items():
            col = BC_COLS[name]
            ax.plot(day_data.index.hour + day_data.index.minute / 60,
                    day_data[col].values, lw=1, alpha=0.8,
                    color=info['color'], label=f"{name}")
        
        ax.set_title(f'{site} — {label}\n{day}', fontsize=10)
        ax.set_xlabel('Hour')
        ax.set_ylabel('BC (µg/m³)' if j == 0 else '')
        ax.set_xlim(0, 24)
        if j == 0 and i == 0:
            ax.legend(fontsize=7)

fig.suptitle('Representative Day Fingerprints', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'representative_days.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Wavelength × Hour Heatmaps & Summary Table

In [ ]:
# --- Normalized heatmaps: wavelength × hour for each site ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes_flat = axes.flatten()
wl_names = list(WAVELENGTHS.keys())

for idx, site in enumerate(sites):
    ax = axes_flat[idx]
    df = site_data[site]
    
    heatmap_data = np.zeros((len(wl_names), 24))
    for i, name in enumerate(wl_names):
        col = BC_COLS[name]
        hourly_med = df.groupby('Hour')[col].median()
        daily_med = hourly_med.mean()
        heatmap_data[i, :] = (hourly_med / daily_med).values if daily_med > 0 else 0
    
    im = ax.imshow(heatmap_data, aspect='auto', cmap='RdYlBu_r',
                   vmin=0.5, vmax=2.0, interpolation='nearest')
    ax.set_yticks(range(len(wl_names)))
    ax.set_yticklabels([f'{n} ({WAVELENGTHS[n]["nm"]}nm)' for n in wl_names])
    ax.set_xticks(range(0, 24, 3))
    ax.set_xlabel('Hour of Day')
    ax.set_title(f'{site}')
    plt.colorbar(im, ax=ax, label='Normalized BC', shrink=0.8)

fig.suptitle('Normalized Diurnal Heatmap (wavelength × hour)', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'heatmap_all_sites.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Summary statistics table ---
time_blocks = {
    'Night (0-5)':     (0, 5),
    'Morning (5-9)':   (5, 9),
    'Midday (9-16)':   (9, 16),
    'Evening (16-21)': (16, 21),
    'Late (21-24)':    (21, 24),
}

rows = []
for site in sites:
    df = site_data[site]
    for season in get_season_list(site) + ['All']:
        sdata = df if season == 'All' else df[df['Season'] == season]
        
        for block_label, (h0, h1) in time_blocks.items():
            block = sdata[(sdata['Hour'] >= h0) & (sdata['Hour'] < h1)]
            ir_valid = block['IR BCc'].notna().sum()
            
            row = {
                'Site': site,
                'Season': season,
                'Time Block': block_label,
                'n_minutes': ir_valid,
                'IR_BCc_median': block['IR BCc'].median(),
                'UV_IR_ratio_median': block['UV/IR'].median() if 'UV/IR' in block.columns else np.nan,
                'AAE_UV_IR_median': block['AAE_UV_IR'].median(),
                'AAE_Blue_IR_median': block['AAE_Blue_IR'].median(),
                'Delta_C_median': block['Delta_C'].median(),
                'wl_CV_median': block['wl_CV'].median(),
            }
            rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(PLOT_DIR / 'multisite_summary_stats.csv', index=False)
print(f"Summary table saved: {len(summary_df)} rows")
print(f"  Sites: {summary_df['Site'].unique()}")
print(f"  Output: {PLOT_DIR / 'multisite_summary_stats.csv'}")

# Display a compact view
display(summary_df[summary_df['Time Block'] == 'Morning (5-9)'][
    ['Site', 'Season', 'n_minutes', 'IR_BCc_median', 'AAE_UV_IR_median', 'Delta_C_median', 'wl_CV_median']
].round(3))